# Ahead Assist — YOLO Explorer (Colab)

Experiment with object detection for the [ahead-assist](https://github.com/adhikariastha5/ahead-assist) project.

**What you'll do:**
- Run YOLO on sample & your own images
- Use the same zone / distance / alert logic as `app.py`
- Compare models and confidence thresholds
- Try ideas before adding them to the web app

> Learning demo only — not a navigation aid.

In [ ]:
# Install deps (Colab usually has torch; ultralytics pulls the rest)
!pip install -q ultralytics opencv-python-headless matplotlib pillow

## 1. Shared detection logic (same as `app.py`)

Edit `PRIORITY`, `CENTER_BAND`, or `CONFIDENCE` here and re-run to experiment.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image
from ultralytics import YOLO

CONFIDENCE = 0.45
IOU = 0.5
CENTER_BAND = (0.2, 0.8)  # horizontal band for "what's ahead"

PRIORITY = {
    "car": 10, "truck": 10, "bus": 10, "motorcycle": 9, "bicycle": 8,
    "person": 7, "dog": 6, "cat": 6, "train": 9,
    "traffic light": 5, "stop sign": 5, "bench": 3,
}


def zone_from_x(cx: float) -> str:
    if cx < 0.33:
        return "left"
    if cx > 0.66:
        return "right"
    return "center"


def distance_hint(area_ratio: float) -> str:
    if area_ratio > 0.25:
        return "very close"
    if area_ratio > 0.12:
        return "close"
    if area_ratio > 0.05:
        return "ahead"
    return "in the distance"


def phrase_for(d: dict[str, Any]) -> str:
    label, zone, dist = d["class"], d["zone"], d["distance"]
    if zone == "center":
        return f"{label} very close ahead" if dist == "very close" else f"{label} ahead, {dist}"
    return f"{label} on your {zone}, {dist}"


def parse_detections(result, frame_shape, center_band=CENTER_BAND) -> list[dict[str, Any]]:
    h, w = frame_shape[:2]
    frame_area = float(h * w)
    out = []
    boxes = result.boxes
    if boxes is None:
        return out
    for box in boxes:
        cls_id = int(box.cls[0])
        name = result.names[cls_id]
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx = ((x1 + x2) / 2) / w
        if not (center_band[0] <= cx <= center_band[1]):
            continue
        area_ratio = ((x2 - x1) * (y2 - y1)) / frame_area
        out.append({
            "class": name,
            "confidence": round(conf, 2),
            "zone": zone_from_x(cx),
            "distance": distance_hint(area_ratio),
            "priority": PRIORITY.get(name, 1),
            "bbox": (x1, y1, x2, y2),
        })
    out.sort(key=lambda d: (-d["priority"], -d["confidence"]))
    return out


def show_detections(rgb: np.ndarray, detections: list[dict], title: str = ""):
    h, w = rgb.shape[:2]
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(rgb)
    # center band
    ax.axvline(w * CENTER_BAND[0], color="#f0a500", linestyle="--", alpha=0.7)
    ax.axvline(w * CENTER_BAND[1], color="#f0a500", linestyle="--", alpha=0.7)
    for d in detections:
        x1, y1, x2, y2 = d["bbox"]
        color = "#f0a500" if d["zone"] == "center" else "#72f1b8"
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 4, 8), f"{d['class']} {d['confidence']}", color="white", fontsize=9,
                bbox=dict(facecolor="black", alpha=0.5, pad=1))
    ax.set_title(title or "Detections in center band")
    ax.axis("off")
    plt.show()

## 2. Load YOLO model

Try `yolov8n.pt` (fast) or `yolov8s.pt` (more accurate, slower).

In [ ]:
MODEL_NAME = "yolov8n.pt"  # change to yolov8s.pt to experiment
model = YOLO(MODEL_NAME)
print("Classes:", len(model.names), "| sample:", list(model.names.values())[:8])

## 3. Run on a sample street image

In [ ]:
from urllib.request import urlretrieve

SAMPLE_URL = "https://ultralytics.com/images/bus.jpg"
sample_path = Path("sample_bus.jpg")
if not sample_path.exists():
    urlretrieve(SAMPLE_URL, sample_path)

bgr = cv2.imread(str(sample_path))
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

results = model.predict(source=bgr, conf=CONFIDENCE, iou=IOU, verbose=False)
detections = parse_detections(results[0], bgr.shape)

print("Alerts (what the web app would say):")
for d in detections[:5]:
    print(" •", phrase_for(d))

show_detections(rgb, detections, title=f"{MODEL_NAME} @ conf={CONFIDENCE}")

## 4. Upload your own image

Take a photo of a street / hallway / room and upload it here.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()  # pick an image in Colab
    upload_name = next(iter(uploaded))
    bgr = cv2.imdecode(np.frombuffer(uploaded[upload_name], np.uint8), cv2.IMREAD_COLOR)
except ImportError:
    # Local Jupyter: set path manually
    upload_name = "your_image.jpg"
    bgr = cv2.imread(upload_name)
    assert bgr is not None, f"Put an image at {upload_name} or run in Colab"

rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
results = model.predict(source=bgr, conf=CONFIDENCE, iou=IOU, verbose=False)
detections = parse_detections(results[0], bgr.shape)

for d in detections[:5]:
    print(phrase_for(d))
show_detections(rgb, detections, title=upload_name)

## 5. Experiment: confidence threshold

Lower confidence = more detections (more false alarms). Higher = fewer, stricter.

In [ ]:
for conf in [0.25, 0.45, 0.65]:
    res = model.predict(source=bgr, conf=conf, iou=IOU, verbose=False)
    dets = parse_detections(res[0], bgr.shape)
    print(f"conf={conf}: {len(dets)} objects in center band")
    if dets:
        print("  top:", phrase_for(dets[0]))

## 6. Experiment: compare nano vs small model

In [ ]:
import time

for name in ["yolov8n.pt", "yolov8s.pt"]:
    m = YOLO(name)
    t0 = time.perf_counter()
    res = m.predict(source=bgr, conf=CONFIDENCE, iou=IOU, verbose=False)
    ms = (time.perf_counter() - t0) * 1000
    dets = parse_detections(res[0], bgr.shape)
    print(f"{name}: {ms:.0f} ms | {len(dets)} detections in band")

## 7. Optional: process a short video

Upload a `.mp4` (5–10 seconds). Samples every 10th frame.

In [ ]:
PROCESS_VIDEO = False  # set True after uploading a video in Colab

if PROCESS_VIDEO:
    from google.colab import files
    vid_upload = files.upload()
    vid_path = next(iter(vid_upload))
    cap = cv2.VideoCapture(vid_path)
    frame_i = 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        if frame_i % 10 == 0:
            res = model.predict(source=frame, conf=CONFIDENCE, verbose=False)
            dets = parse_detections(res[0], frame.shape)
            if dets:
                print(f"frame {frame_i}:", phrase_for(dets[0]))
        frame_i += 1
    cap.release()
else:
    print("Set PROCESS_VIDEO = True and upload an mp4 to try this.")

## 8. Ideas to try next

- Narrow `CENTER_BAND` to `(0.35, 0.65)` for stricter "ahead" zone
- Add classes to `PRIORITY` (e.g. `stairs` if you fine-tune later)
- Filter by vertical position (ignore sky / ground)
- Export best `CONFIDENCE` back to `app.py`
- Fine-tune YOLO on wheelchair-relevant objects (advanced, needs dataset)